# F1 data: read the table before the model
IIT414W · Week 1 · Friday 4 September 2026 · Student notebook v1

Today we continue the reproducible environment from Thursday. We obtain two tables, explain their rows and fields, and inspect a few checks. **No previous F1 knowledge is required.** We do not train a model today.

Unit outcomes practised (not fully assessed in this session):
- Configure a reproducible machine learning environment, implementing version control, dependency management, and fixed random seeds.
- Evaluate data quality and design appropriate train/validation/test splits to prevent data leakage and ensure robust temporal validation.

**Case:** the 2021 Italian Grand Prix at Monza. The year is within the course training period. Synthetic fallback records have a different, explicit label; they are not Monza data.

**Materials:** the full course code folder, Python/Jupyter, pandas; requests and FastF1 for the live route. Keep `w01_fri_support_v1.py` next to this notebook. Open the runbook if a dependency is missing. Never install packages silently during Run All.

**Your evidence:** your own tables, their provenance, computed checks, a short dictionary response and a next action. Working with a partner is allowed; each person keeps their own explanation.

Read each instruction before running its code. Pause and ask for a reformulation when needed. Times guide the group; they are not speed grades.

## Route through the studio
| Block | What you do | Working time | Expected evidence |
|---|---|---:|---|
| 1 · Results | Check your environment; read the Jolpica table | 20 min | Provenance and one-row interpretation |
| Pause | Stop coding and take the class break | 10 min | No task |
| 2 · Laps | Load the same case with FastF1; compare granularity | 25 min | Two distinct row definitions |
| 3 · Save | Export tables and checks | 5 min | A run folder with a manifest |
| 4 · Dictionary clinic | Explain fields, review one check, use feedback | 25 min | Your individual written record |

These blocks occupy 85 minutes within the 150-minute class. The opening, Lab 0 briefing and exit ticket are separate. Read the next block only after finishing or explicitly recording your blocker.

## Short glossary
- **Season:** one year's championship. **Circuit:** the track. **Grand Prix/race:** an event at that track.
- **Driver / constructor:** the person driving / their team. Names are labels; stable identifiers make better keys.
- **Session:** one scheduled activity, such as qualifying (`Q`) or the race (`R`).
- **Lap:** one circuit of the track. Many laps belong to one driver in one race.
- **Grid:** recorded starting-place field. Zero is a special/unspecified start code, not “better than first”; investigate before calculating gains.
- **Classification / status:** recorded finishing rank / description of the outcome. A numeric rank alone does not establish that a driver finished the race.
- **Compound:** tyre category. **Telemetry:** a sequence of car measurements; we do not download it today.
- **API:** a service queried by code. **Cache:** data stored by a library for reuse. **Snapshot:** a provided copy with a recorded source and hash.
- **Provenance:** where these data came from. **Grain:** what one row represents. **Hash:** a fingerprint used to detect byte changes.
- **Kernel:** the Python process behind this notebook. **Seed:** a fixed starting state for controlled randomness, not a guarantee that all results are correct.

## 1 · Start from your own environment
**Read first.** Check Thursday's Ready / Minor fix / Blocked status. Run the next two cells. They do not install anything or alter Git.

**Before loading, write your own answer:**
- What would count as a successful data load, beyond “the cell ran”? **Your answer:**
- What might block it? **Your answer:**

**Pause point:** if you cannot find the full project folder, ask before changing paths.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / ".iit414w-root").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the full course code folder containing .iit414w-root. No folders were created.")
WEEK = ROOT / "unit_I/week_01"
if str(WEEK) not in sys.path:
    sys.path.insert(0, str(WEEK))
import w01_fri_support_v1 as support
from IPython.display import display
pd = support.require_pandas()
RANDOM_SEED = 414
display(pd.DataFrame(support.versions().items(), columns=["item", "observed"]))

,item,observed
0,python,3.13.7
1,seed,414
2,git_available,True
3,numpy,2.3.4
4,pandas,2.3.3
5,requests,2.32.5
6,fastf1,MISSING


### Choose a data route deliberately
`live` tries the APIs/library, then a verified provided snapshot, then synthetic data if neither works. Every fallback prints its status. `snapshot` does not request network data. `synthetic` needs no network or real-data files.

**How:** keep `live` for the guided first attempt. If the connection stalls, interrupt the kernel, choose `snapshot`, and restart from the first code cell. The lecturer will demonstrate the live code if your own machine cannot connect.

**Expected evidence:** the actual mode reported for each table. A snapshot or synthetic run does **not** demonstrate successful API access.

In [5]:
MODE = "live"  # Allowed: "live", "snapshot", "synthetic"
results, results_source = support.load_table(ROOT, "results", MODE)
print("Actual results source:", results_source)
display(results[["driver_id", "constructor_id", "grid", "position", "points", "status"]].head(8))
print("Results shape:", results.shape)

results: JOLPICA_HTTP
Actual results source: {'origin': 'JOLPICA_HTTP', 'api_access_confirmed': True, 'url': 'https://api.jolpi.ca/ergast/f1/2021/circuits/monza/results/', 'pages': 1, 'total': 20, 'retrieved_utc': '2026-09-04T16:07:49.475834+00:00'}


,driver_id,constructor_id,grid,position,points,status
0,ricciardo,mclaren,2,1,26,Finished
1,norris,mclaren,3,2,18,Finished
2,bottas,mercedes,19,3,15,Finished
3,leclerc,ferrari,5,4,12,Finished
4,perez,red_bull,8,5,10,Finished
5,sainz,ferrari,6,6,8,Finished
6,stroll,aston_martin,9,7,6,Finished
7,alonso,alpine,10,8,4,Finished


Results shape: (20, 11)


### What the helper did
For Jolpica, the visible helper requests HTTPS, checks HTTP status, reads `MRData.total`, `limit` and `offset`, and rejects incomplete pages. It flattens JSON into one row per driver per race. It keeps identifiers and does not replace missing values with zero.

Open `w01_fri_support_v1.py` and find `fetch_jolpica` and `normalize_results`. You may ask AI to explain permitted code, but verify any claim against the code and your own output. Do not manufacture an AI interaction.

**Your turn (within the first 20-minute block):**
1. Select one row in the displayed table.
2. Write what that row represents, and name two of its fields.
3. Record the actual source: HTTP, provided snapshot or synthetic.

**Your row and explanation:**

Let's focus on the third row, this one represents the resume of Bottas's race, he amazingly started 19 (grid) and finished 3 (position).

**Actual source and limitation:**

The source is HTTP, so we're actually downloading data directly from the API.

**Pause point:** a dataframe can be readable and still have the wrong grain. Ask before treating its rows as laps.

## Class break
Stop here for the planned 10-minute break. The next block uses a different table, not a second prediction exercise.

## 2 · Read the lap table
**Task:** obtain lap-level data for the same teaching case and compare it with the results table. **How:** run the cell, then inspect the row count and the key columns. **Time:** 25 minutes including checks. **Evidence:** a correct row definition and your explanation of one check.

The live helper runs:
```python
session = fastf1.get_session(2021, "Italy", "R")
session.load(laps=True, telemetry=False, weather=False, messages=False)
```
Only lap timing is requested. FastF1 may use its cache; successful loading does not prove a new network request. If it stalls, use the runbook fallback. Do not repeatedly download sessions.

In [ ]:
laps, laps_source = support.load_table(ROOT, "laps", MODE)
print("Actual lap source:", laps_source)
display(laps[["driver_number", "lap_number", "lap_time_s", "compound", "is_accurate"]].head(8))
print("Lap table shape:", laps.shape)
print("Results source:", results_source["origin"], "| Laps source:", laps_source["origin"])
print("Do not merge real and synthetic tables as though they describe the same event.")

### Compare grain before comparing values
**Your results-table row definition:**

**Your lap-table row definition:**

**Why can one driver appear many times in the lap table?**

**Which columns would identify one lap within this season and race?**

No prior knowledge of drivers or teams is needed: use the field names and dictionary. Missing lap time is a reason to investigate, not an instruction to drop the row automatically.

In [ ]:
checks = support.quality_checks(results, laps)
display(checks)
print("PASS checks structure only. It does not prove API access, prediction validity or understanding.")

### Your independent check
**How:** add one simple computed check below, or repeat one existing check and explain why it matters. Examples of questions to investigate: Is each table limited to one circuit? Are driver numbers present? Do points have missing values?

Do not type a literal `True` or a guessed count as a check. Compute from the table. A failure can be useful evidence.

**What I check and why:**

**My observed result:**

**What this result does not prove:**

In [ ]:
# YOUR TURN: write your check here. This blank cell intentionally runs without producing an answer.
# Do not replace a computed result with a hardcoded PASS.

### A result is not a pre-race feature
Suppose you want to predict a driver's finishing position **just before the race starts**.

In your dictionary response, classify two candidate fields by when they become known. Do not build a model.

**Field 1 / known when / usable at the stated prediction time / reason:**

**Field 2 / known when / usable at the stated prediction time / reason:**

For later assessed modelling, the course split remains train through 2021, calibration 2022, test 2023–2024. Today's descriptive inspection is not model selection or test evaluation. A final-season standings table contains information from later races and must not be treated as an earlier pre-race feature.

## 3 · Save what actually happened
**Task:** export both tables, the computed checks and provenance. **How:** run the next cell; note the relative folder printed. **Time:** 5 minutes. **Evidence:** CSV files and a manifest with hashes and environment versions.

Every run creates a new folder. It does not overwrite previous evidence. The manifest does not fill in your personal explanations or claim that you restarted the kernel.

In [ ]:
RUN_FOLDER = support.export_evidence(
    ROOT, results, laps,
    {"results": results_source, "laps": laps_source}, checks
)
print("Manifest:", (RUN_FOLDER / "run_manifest.json").relative_to(ROOT).as_posix())

## 4 · Dictionary clinic and individual check
**Materials:** your tables and `W01_Fri_DataDictionary_v1.md`. **Time:** 25 minutes. Read the whole instruction, then work independently while the lecturer checks each person briefly. You may ask for clarification.

1. Complete your row definitions and two field explanations above.
2. Explain one computed check using your own output.
3. Show one field that cannot be known at the prediction time and explain why.
4. Record one correction you make after feedback. If no correction is needed, explain what was checked and keep a concrete next step.

**Feedback received:**

**Change / recheck / observed result:**

**What I will document in my repository and runbook:**

Use the remaining clinic time to restart the kernel and run all cells. Compare the exported tables from two runs; file timestamps/folder names are expected to differ. With an unchanged provided snapshot, data hashes should match. Live data may be revised by its provider; document differences rather than forcing equality.

The lecturer checks your explanation, not your knowledge of F1. This studio does not add a graded oral test.

## Before the Lab 0 briefing
- Save your own notebook with your explanations.
- Use the existing PROMPTS template for significant AI assistance, or state honestly that you did not use AI.
- Do not include secrets, private information or invented outputs.
- Lab 0: individual, 3% of NP, due **Thursday 10 September at 12:30**. Read the separate briefing.
- Pre-Course Diagnostic: due **today, Friday 4 September, at 23:59**.

The exit ticket is separate. Follow the published Canvas policy for participation bonus; this notebook does not redefine it.

## Sources
- [Jolpica documentation and pagination](https://github.com/jolpica/jolpica-f1/blob/main/docs/README.md)
- [Jolpica result fields](https://github.com/jolpica/jolpica-f1/blob/main/docs/endpoints/results.md)
- [FastF1 project and documentation](https://github.com/theOehrly/Fast-F1)
- Course syllabus and Academic Schedule T3 2026.

Read the data provenance in your own run; these documentation links are not evidence that your API request succeeded.